# Hybrid MESH calibration: Witt (1998) blended with Murillo & Homeyer (2019)

This notebook defines a **smoothly blended** Maximum Expected Size of Hail (MESH)
formulation that uses Witt (1998) below the analytical intercept of the two power
laws and Murillo & Homeyer (2019) above it, with a logistic weight providing a
$C^{\infty}$ handoff between them.

$$\text{MESH}(\text{SHI}) = (1-w)\cdot 2.54\,\text{SHI}^{0.5} + w\cdot 15.096\,\text{SHI}^{0.206}$$

$$w(\text{SHI}) = \frac{1}{1 + \exp\!\big(-k\,(\text{SHI} - \text{SHI}_*)\big)},\qquad k = \frac{2\ln 9}{\text{transition\_width}}$$

**Requirements**: `numpy`, `matplotlib`, `ipywidgets`. All are typically already
available in a scientific Python environment; if not:

```bash
pip install numpy matplotlib ipywidgets
```

Then in VS Code, open this `.ipynb` and select a Python kernel that has these packages.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

## Calibration constants and analytical intercept

Setting $2.54\,\text{SHI}^{0.5} = 15.096\,\text{SHI}^{0.206}$ gives
$\text{SHI}_* = (15.096/2.54)^{1/(0.5-0.206)} \approx 429.05$ J m$^{-1}$ s$^{-1}$,
with both calibrations crossing at MESH $\approx 52.6$ mm.

In [ ]:
# Calibration constants
WITT_A, WITT_B = 2.54, 0.5            # Witt (1998)
MH19_A, MH19_B = 15.096, 0.206        # Murillo & Homeyer 75th percentile (2019)

# Analytical intercept of the two power laws (J m^-1 s^-1)
SHI_INTERCEPT = (MH19_A / WITT_A) ** (1.0 / (WITT_B - MH19_B))
MESH_INTERCEPT = WITT_A * SHI_INTERCEPT ** WITT_B

print(f"SHI*  = {SHI_INTERCEPT:8.3f}  J m^-1 s^-1")
print(f"MESH* = {MESH_INTERCEPT:8.3f}  mm")

SHI*  =  429.288  J m^-1 s^-1
MESH* =   52.627  mm


## Smooth-blend implementation

The `transition_width` argument controls the SHI range over which the weight
$w$ moves from 0.1 to 0.9. Smaller values tend toward the hard piecewise limit;
larger values give a gentler handoff that biases values either side of the pivot.

In [23]:
def mesh_smooth_blend(shi, transition_width=200.0, shi_pivot=SHI_INTERCEPT):
    """
    Smoothly blended MESH (mm), Witt (1998) -> Murillo & Homeyer (2019).

    Parameters
    ----------
    shi : array_like
        Severe Hail Index, J m^-1 s^-1. Non-positive values return 0.
    transition_width : float, optional
        SHI range over which the logistic weight transitions from 0.1 to 0.9.
    shi_pivot : float, optional
        SHI value at which w = 0.5 (default = analytical intercept).

    Returns
    -------
    mesh : ndarray
        MESH in mm, same shape as `shi`.
    """
    shi = np.asarray(shi, dtype=float)

    # Logistic steepness: w goes from 0.1 to 0.9 across `transition_width`.
    k = 2.0 * np.log(9.0) / float(transition_width)

    # Clip to avoid overflow at extreme SHI on large grids.
    z = np.clip(-k * (shi - shi_pivot), -50.0, 50.0) #clip to reduce numerical noise in exp(): 1/(1+exp(-50))~0 and 1/(1+exp(50))~1
    w = 1.0 / (1.0 + np.exp(z))

    safe_shi = np.maximum(shi, 0.0)
    mesh_witt = WITT_A * np.power(safe_shi, WITT_B)
    mesh_mh19 = MH19_A * np.power(safe_shi, MH19_B)

    mesh = (1.0 - w) * mesh_witt + w * mesh_mh19
    return np.where(shi > 0.0, mesh, 0.0)

## Sanity check

At the pivot the blend should equal both calibrations exactly. With a very small
`transition_width` the blend should also approach the hard piecewise definition.

In [27]:
shi_test = np.array([50.0, 200.0, SHI_INTERCEPT, 700.0, 1500.0])

mesh_witt_only = WITT_A * shi_test ** WITT_B
mesh_mh19_only = MH19_A * shi_test ** MH19_B
mesh_hard = np.where(shi_test <= SHI_INTERCEPT, mesh_witt_only, mesh_mh19_only)
mesh_blend = mesh_smooth_blend(shi_test, transition_width=200.0)
mesh_tight = mesh_smooth_blend(shi_test, transition_width=10.0)  # ~ hard

print(f"{'SHI':>10}{'Witt':>10}{'MH19':>10}{'Hard':>10}{'blend_200':>12}{'blend_10':>10}")
for s, w_, m_, h_, b_, t_ in zip(shi_test, mesh_witt_only, mesh_mh19_only,
                                  mesh_hard, mesh_blend, mesh_tight):
    print(f"{s:>10.1f}{w_:>10.2f}{m_:>10.2f}{h_:>10.2f}{b_:>12.2f}{t_:>10.2f}")

       SHI      Witt      MH19      Hard   blend_200  blend_10
      50.0     17.96     33.79     17.96       17.96     17.96
     200.0     35.92     44.96     35.92       35.98     35.92
     429.3     52.63     52.63     52.63       52.63     52.63
     700.0     67.20     58.20     58.20       58.23     58.20
    1500.0     98.37     68.10     68.10       68.10     68.10


## Interactive exploration

Drag the slider to see how `transition_width` reshapes the blended curve relative
to the two underlying calibrations. The first execution may take a moment while
ipywidgets initialises in VS Code.

In [20]:
shi_grid = np.linspace(1.0, 1200.0, 600)
witt_curve = WITT_A * shi_grid ** WITT_B
mh19_curve = MH19_A * shi_grid ** MH19_B

@interact(transition_width=FloatSlider(
    min=1.0, max=400.0, step=1.0, value=200.0,
    description='Width', continuous_update=False))
def _plot_blend(transition_width=100.0):
    blend = mesh_smooth_blend(shi_grid, transition_width=transition_width)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(shi_grid, witt_curve, '--', color='gray',
            label=r'Witt 1998: $2.54\,\mathrm{SHI}^{0.5}$', lw=1.2)
    ax.plot(shi_grid, mh19_curve, '--', color='steelblue',
            label=r'M&H 2019: $15.096\,\mathrm{SHI}^{0.206}$', lw=1.2)
    ax.plot(shi_grid, blend, color='firebrick',
            label=f'Smooth blend (width = {transition_width:.0f})', lw=2.2)
    ax.scatter([SHI_INTERCEPT], [MESH_INTERCEPT], color='firebrick',
               zorder=5, label=f'Pivot (SHI = {SHI_INTERCEPT:.1f})')

    ax.set_xlabel(r'SHI (J m$^{-1}$ s$^{-1}$)')
    ax.set_ylabel('MESH (mm)')
    ax.set_xlim(50, 700)
    ax.set_ylim(10, 70)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right', fontsize=10, framealpha=0.95)
    ax.set_title('Hybrid MESH formulation: smooth blend')
    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=200.0, continuous_update=False, description='Width', max=400.0, min=1.…

Inspect the derivative

The smoothing's main benefit is removing the kink in $d\text{MESH}/d\text{SHI}$.
Numerical differentiation makes this easy to see.

In [21]:
def plot_derivative(transition_width=200.0):
    shi = np.linspace(1.0, 1200.0, 4000)
    mesh_blend = mesh_smooth_blend(shi, transition_width=transition_width)
    mesh_hard = np.where(shi <= SHI_INTERCEPT,
                         WITT_A * shi ** WITT_B,
                         MH19_A * shi ** MH19_B)
    d_blend = np.gradient(mesh_blend, shi)
    d_hard = np.gradient(mesh_hard, shi)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(shi, d_hard, color='gray', label='Hard piecewise', lw=1.2)
    ax.plot(shi, d_blend, color='firebrick',
            label=f'Smooth blend (width = {transition_width:.0f})', lw=2.0)
    ax.axvline(SHI_INTERCEPT, color='k', ls=':', alpha=0.4)
    ax.set_xlabel(r'SHI (J m$^{-1}$ s$^{-1}$)')
    ax.set_ylabel(r'$d\mathrm{MESH}/d\mathrm{SHI}$ (mm per unit SHI)')
    ax.set_xlim(50, 700)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=10)
    ax.set_title('Derivative of MESH wrt SHI')
    plt.tight_layout()
    plt.show()

interact(plot_derivative, transition_width=FloatSlider(
    min=1.0, max=400.0, step=1.0, value=100.0,
    description='Width', continuous_update=False));

interactive(children=(FloatSlider(value=100.0, continuous_update=False, description='Width', max=400.0, min=1.…